In [ ]:
import os
from src.utils import env_utils
from src.utils.experiment_utils import set_seed
from src.probing.prompt import prepare_choice_input
from src.tokens import TokenizerOutput
from src.functional import interpret_logits

probe_dir = os.path.join(env_utils.DEFAULT_DATA_DIR, "probe")

os.makedirs(probe_dir, exist_ok=True)

# Iterate through attributes. Just profession in this case
for attribute in probe_pairs.keys():
    print("#" * 50)
    print(f"{attribute}")
    print("#" * 50)

    prompt_template = """Q:Which of the following people has a profession in common with {}? {}.\nA:"""

    set_seed(9001)

    # Locations is just the last token position of each layer & of the lm_head
    locations = [(layer_name, -1) for layer_name in mt.layer_names]
    lm_head_state = (mt.lm_head_name, -1)
    locations += [lm_head_state]
    locations = list(set(locations))

    # Create a directory to store the data for this attribute
    attribute_dir = os.path.join(probe_dir, attribute)
    os.makedirs(attribute_dir, exist_ok=True)

    # Loop through the specific profession, and the corresponding entities
    for shared_attr, entities in probe_pairs[attribute].items():
        print("-" * 50)
        print(f"Shared attribute: {shared_attr}")
        print("-" * 50)

        # List to store cached hidden states
        cached_states: list[CachedStates] = []

        # Make quartets out of entities to emulate the subject_entity, common_entity scheme for the choose one task
        # Loop through these quartets
        for current_query_quartet in list(itertools.combinations(entities, 4)):
            current_query_quartet = list(current_query_quartet)

            # This formats the choose one task prompt with the quartet entities
            # NOTE: May be a bad implementation, the current setup can contain relations of between 0 and 4 rather than just 2 like the main task
            current_input = prepare_choice_input(
                mt=mt,
                clean_entity=current_query_quartet[0],
                common_entities=current_query_quartet[1:],
                prompt_template=prompt_template
            )

            # Get the hidden state at the last token position of each layer and the lm_head
            probe_hs = get_hs(
                mt=mt,
                input=TokenizerOutput(data=current_input.tokenized),
                locations=locations,
                return_dict=True
            )

            # Calculate the logits at the lm_head
            logits = probe_hs[(mt.lm_head_name, -1)]

            # Get the top token predictions
            cur_top_preds = interpret_logits(
                tokenizer=mt,
                logits=logits,
                k=15
            )

            # Format the top predictions
            ll_fmt = [
                f'"{pred.token}"[p={pred.prob:.2f}, l={pred.logit:.2f}]'
                for pred in cur_top_preds
            ]
            print(f"{current_query_quartet} => {ll_fmt[:5]}")

            # Append these hidden states to our list
            cached_states.append(
                CachedStates(
                    subj_entity=current_query_quartet[0],
                    common_entities=current_query_quartet[1:],
                    states=probe_hs,
                    model_predictions=cur_top_preds,
                )
            )

        # Set up a dictionary to hold probe directions for each layer
        probe_directions = {
            layer_name: [] for layer_name in mt.layer_names
        }

        # Make the cached states at each layer our probe directions for that layer
        for cached_state in cached_states:
            for layer_name in mt.layer_names:
                probe_directions[layer_name].append(
                    cached_state.states[(layer_name, -1)]
                )

        # The probe direction is a matrix of directions for each layer
        # for each layer and direction in the probe_directions dictionary
        probe_directions = ProbeDirection(
            directions={
                layer_name: torch.stack(directions, dim=0).mean(dim=0)
                for layer_name, directions in probe_directions.items()
            },
            metadata=cached_states,
        )

        # Save the probe directions
        probe_directions.detensorize()
        probe_path = os.path.join(
            attribute_dir, f"{shared_attr}.json"
        )
        print("saving to >>", probe_path)
        with open(probe_path, "w") as f:
            f.write(probe_directions.to_json())
    
    print()